In [45]:
! pip3 install tiktoken

In [46]:
tokenizer=tiktoken.get_encoding("gpt2") #usage of this is a same as  a SimpleTokenizerV2

In [47]:
text=(
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace"
)
integers=tokenizer.encode(text,allowed_special={"<|endoftext|>"})
print("Encoded:",integers)

Encoded: [15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271]


In [48]:
strings=tokenizer.decode(integers)
print("Decoded:",strings)

Decoded: Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace


In [49]:
integer=tokenizer.encode("lalalalalal djgbhjbdfs")
print("Single Encoded:",integer)

decoded=tokenizer.decode(integer)
print("Single Decoded:",decoded)

Single Encoded: [75, 282, 282, 282, 282, 282, 42625, 22296, 71, 73, 65, 7568, 82]
Single Decoded: lalalalalal djgbhjbdfs


In [50]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text=f.read()

enc_text=tokenizer.encode(raw_text)
print(f"Length of the text in tokens: {len(enc_text)}"  )

Length of the text in tokens: 5145


In [51]:
enc_sample=enc_text[50:]

In [52]:
context_size=4 #length of the input
x=enc_sample[:context_size] #first 4 tokens - input for llm
y=enc_sample[1:context_size+1] #next 4 tokens - and what output is
print("Input tokens:",x)
print(f"Target tokens:     {y}")

Input tokens: [290, 4920, 2241, 287]
Target tokens:     [4920, 2241, 287, 257]


In [53]:
for i in range (1,context_size+1):
    context=enc_sample[:i]
    desired=enc_sample[i]
    print(f"Input tokens:{context} -> Target token:{desired}")

Input tokens:[290] -> Target token:4920
Input tokens:[290, 4920] -> Target token:2241
Input tokens:[290, 4920, 2241] -> Target token:287
Input tokens:[290, 4920, 2241, 287] -> Target token:257


In [54]:
for i in range(1,context_size+1):
    context=enc_sample[:i]
    desired=enc_sample[i]

    print(tokenizer.decode(context), "->", tokenizer.decode([desired]))

 and ->  established
 and established ->  himself
 and established himself ->  in
 and established himself in ->  a


In [ ]:
class GPTDatasetV1:
    def __init__(self,txt,tokenizer,max_length,stride): # what stride is epxlain simply take example ? ans - it is how much we move forward for next sample
        self.input_ids=[]
        self.target_ids=[]

        #tokenized the entire text
        token_ids=tokenizer.encode(txt,allowed_special={"<|endoftext|>"})

        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk=token_ids[i:i + max_length]
            target_chunk=token_ids[i+1:i+max_length+1]
            # store raw lists to avoid torch dependency here
            self.input_ids.append(input_chunk)
            self.target_ids.append(target_chunk)

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self,idx):
        return self.input_ids[idx],self.target_ids[idx]

In [68]:
import random

class SimpleDataLoader:
    def __init__(self, dataset, batch_size=4, shuffle=True, drop_last=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.drop_last = drop_last

    def __iter__(self):
        idxs = list(range(len(self.dataset)))
        if self.shuffle:
            random.shuffle(idxs)
        limit = len(idxs)
        if self.drop_last:
            limit = (limit // self.batch_size) * self.batch_size
        for start in range(0, limit, self.batch_size):
            batch_idxs = idxs[start:start + self.batch_size]
            xs = []
            ys = []
            for i in batch_idxs:
                x, y = self.dataset[i]
                xs.append(x)
                ys.append(y)
            yield xs, ys


def create_dataloader_v1(txt,batch_size=4,max_length=256,stride=128,shuffle=True,drop_last=True,num_workers=0):

    #initialize the tokenizer
    #stride means number of batch we ignore before we going to new batch
    tokenizer=tiktoken.get_encoding("gpt2")

    #crete a  datset

    dataset=GPTDatasetV1(txt,tokenizer,max_length,stride)

    #create a dataloader (pure-Python, no torch required)
    dataloader = SimpleDataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
    )
    return dataloader


In [62]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text=f.read()

In [ ]:
dataloader=create_dataloader_v1(
    raw_text,batch_size=256,max_length=4,stride=1,shuffle=False
)

data_iter=iter(dataloader)
first_batch=next(data_iter)
print(first_batch)

([[40, 367, 2885, 1464], [367, 2885, 1464, 1807], [2885, 1464, 1807, 3619], [1464, 1807, 3619, 402], [1807, 3619, 402, 271], [3619, 402, 271, 10899], [402, 271, 10899, 2138], [271, 10899, 2138, 257], [10899, 2138, 257, 7026], [2138, 257, 7026, 15632], [257, 7026, 15632, 438], [7026, 15632, 438, 2016], [15632, 438, 2016, 257], [438, 2016, 257, 922], [2016, 257, 922, 5891], [257, 922, 5891, 1576], [922, 5891, 1576, 438], [5891, 1576, 438, 568], [1576, 438, 568, 340], [438, 568, 340, 373], [568, 340, 373, 645], [340, 373, 645, 1049], [373, 645, 1049, 5975], [645, 1049, 5975, 284], [1049, 5975, 284, 502], [5975, 284, 502, 284], [284, 502, 284, 3285], [502, 284, 3285, 326], [284, 3285, 326, 11], [3285, 326, 11, 287], [326, 11, 287, 262], [11, 287, 262, 6001], [287, 262, 6001, 286], [262, 6001, 286, 465], [6001, 286, 465, 13476], [286, 465, 13476, 11], [465, 13476, 11, 339], [13476, 11, 339, 550], [11, 339, 550, 5710], [339, 550, 5710, 465], [550, 5710, 465, 12036], [5710, 465, 12036, 11], [